# PLV + Process+ Exploratory Leaderboards

Unofficial public-data clone. Not affiliated with Pitcher List.

**What this notebook covers:**
1. Hitter leaderboard — Process+, Decision+, Contact+, Power+ with surface stats
2. Pitcher leaderboard — PLV by pitcher and pitch type
3. Fantasy targeting — identifying hitters with strong Process but weak surface stats
4. Rolling trends — who is hot / cold over the last 30 days
5. Component breakdowns — what's driving each hitter's score

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from plv_clone.config import get_config

cfg = get_config()
YEAR = 2024  # change this to score a different season

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print(f'Config loaded. Outputs dir: {cfg.outputs_dir}')

## 1. Load Leaderboards

In [ ]:
# Hitter master leaderboard (includes surface stats)
master_hitter_path = cfg.outputs_dir / f'master_hitter_{YEAR}.csv'
if master_hitter_path.exists():
    hitters = pd.read_csv(master_hitter_path)
else:
    # Fall back to plain process+ leaderboard
    hitters = pd.read_csv(cfg.outputs_dir / f'process_plus_leaderboard_{YEAR}.csv')
    print('Note: master_hitter not found. Run: plv build-exports', YEAR)

# Pitcher leaderboard
pitcher_path = cfg.outputs_dir / f'master_pitcher_{YEAR}.csv'
if pitcher_path.exists():
    pitchers = pd.read_csv(pitcher_path)
else:
    pitchers = pd.read_csv(cfg.outputs_dir / f'plv_leaderboard_{YEAR}.csv')

print(f'Hitters: {len(hitters)} qualified | Pitchers: {len(pitchers)} qualified')
print(f'Hitter columns: {hitters.columns.tolist()}')

## 2. Hitter Leaderboard — Top 25 Process+

In [ ]:
name_col = 'batter_name' if 'batter_name' in hitters.columns else 'batter'
display_cols = [name_col, 'pa', 'process_plus', 'decision_plus', 'contact_plus', 'power_plus']
for col in ('swing_pct', 'chase_pct', 'xwoba_actual', 'xwoba_vs_expected'):
    if col in hitters.columns:
        display_cols.append(col)

top25 = hitters.nlargest(25, 'process_plus')[display_cols]
top25.index = range(1, len(top25) + 1)
top25

## 3. Component Distribution — All Qualified Hitters

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
components = [
    ('process_plus', 'Process+', '#9C27B0'),
    ('decision_plus', 'Decision+', '#2196F3'),
    ('contact_plus',  'Contact+',  '#4CAF50'),
    ('power_plus',    'Power+',    '#FF9800'),
]
for ax, (col, label, color) in zip(axes.flat, components):
    if col not in hitters.columns:
        continue
    vals = hitters[col].dropna()
    ax.hist(vals, bins=30, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(100, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'{label}  (mean={vals.mean():.1f}, std={vals.std():.1f})')
    ax.set_xlabel('+metric')
    ax.set_ylabel('Hitters')

fig.suptitle(f'{YEAR} Process+ Components — {len(hitters)} qualified hitters', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Fantasy Targeting — Strong Process, Weak Surface Stats

These hitters have above-average Process+ but below-median xwOBA.
This can indicate hitters who are "running bad" on contact outcomes
despite making good decisions and quality contact.

In [ ]:
if 'xwoba_actual' in hitters.columns:
    median_xwoba = hitters['xwoba_actual'].median()
    targets = hitters[
        (hitters['process_plus'] >= 105) &
        (hitters['xwoba_actual'] < median_xwoba)
    ].copy()
    targets['xwoba_vs_median'] = (targets['xwoba_actual'] - median_xwoba).round(3)
    cols = [name_col, 'pa', 'process_plus', 'decision_plus', 'power_plus', 'xwoba_actual', 'xwoba_vs_median']
    cols = [c for c in cols if c in targets.columns]
    print(f'Strong Process+ but below-median xwOBA (n={len(targets)})')
    display(targets[cols].sort_values('process_plus', ascending=False).head(20))
else:
    print('xwoba_actual not available. Run build-exports to generate master_hitter.')

## 5. Discipline Leaders — Decision+ Top 20

In [ ]:
disc_cols = [name_col, 'pa', 'decision_plus', 'contact_plus', 'power_plus', 'process_plus']
if 'swing_pct' in hitters.columns:
    disc_cols.append('swing_pct')
if 'chase_pct' in hitters.columns:
    disc_cols.append('chase_pct')

disc_cols = [c for c in disc_cols if c in hitters.columns]
top_disc = hitters.nlargest(20, 'decision_plus')[disc_cols]
top_disc.index = range(1, len(top_disc) + 1)
print('Decision+ Leaders (swing/take decision quality)')
top_disc

## 6. Power Leaders — Power+ Top 20

In [ ]:
pow_cols = [name_col, 'pa', 'power_plus', 'decision_plus', 'contact_plus', 'process_plus']
if 'xwoba_actual' in hitters.columns:
    pow_cols.append('xwoba_actual')
if 'xwoba_vs_expected' in hitters.columns:
    pow_cols.append('xwoba_vs_expected')

pow_cols = [c for c in pow_cols if c in hitters.columns]
top_power = hitters.nlargest(20, 'power_plus')[pow_cols]
top_power.index = range(1, len(top_power) + 1)
print('Power+ Leaders (xwOBA above pitch expectation)')
top_power

## 7. Pitcher Leaderboard — Top 25 PLV

In [ ]:
pitch_name_col = 'player_name' if 'player_name' in pitchers.columns else 'pitcher'
pitch_cols = [pitch_name_col, 'pitches', 'plv', 'swing_pct', 'whiff_pct', 'cs_pct', 'xwoba_model']
pitch_cols = [c for c in pitch_cols if c in pitchers.columns]

top_pitchers = pitchers.nlargest(25, 'plv')[pitch_cols]
top_pitchers.index = range(1, len(top_pitchers) + 1)
top_pitchers

## 8. Rolling Trends — Process+ Last 30 Days

Hitters with the highest average Process+ over their most recent 30-day window.

In [ ]:
rolling_path = cfg.outputs_dir / f'process_plus_rolling_{YEAR}.csv'
if rolling_path.exists():
    rolling = pd.read_csv(rolling_path, parse_dates=['date'])

    # Most recent window for each hitter
    latest = rolling.sort_values('date').groupby('batter').last().reset_index()

    # Join names from hitters leaderboard
    if name_col in hitters.columns:
        names_df = hitters[['batter', name_col]].drop_duplicates()
        latest = latest.merge(names_df, on='batter', how='left')
        latest[name_col] = latest[name_col].fillna(latest['batter'].astype(str))

    print(f'Rolling data: {len(rolling)} hitter-window rows | latest date: {rolling["date"].max()}')

    # Latest 30-day decision and power leaders
    if 'decision_value_mean' in latest.columns:
        display_rolling = [name_col, 'date', 'pa', 'pitches', 'decision_value_mean',
                           'contact_value_mean', 'power_value_mean']
        display_rolling = [c for c in display_rolling if c in latest.columns]
        latest_sorted = latest.sort_values('decision_value_mean', ascending=False)
        print('\nMost recent 30-day Decision+ leaders (raw value, higher = better):')
        display(latest_sorted[display_rolling].head(15))
else:
    print('Rolling data not found. Run: plv build-exports', YEAR)

## 9. Single Hitter Breakdown

Enter a batter ID or name to see their component profile.

In [ ]:
# Change this to inspect a different hitter
HITTER_QUERY = 'Aaron Judge'   # or a numeric batter MLBAM ID

def lookup_hitter(query, df, name_col):
    if isinstance(query, int) or (isinstance(query, str) and query.isdigit()):
        row = df[df['batter'] == int(query)]
    else:
        row = df[df[name_col].str.lower().str.contains(query.lower(), na=False)]
    return row

row = lookup_hitter(HITTER_QUERY, hitters, name_col)
if len(row) == 0:
    print(f'Not found: {HITTER_QUERY}')
else:
    row = row.iloc[0]
    print(f"\n{'='*50}")
    print(f"  {row.get(name_col, row['batter'])}  |  {YEAR}")
    print(f"{'='*50}")
    print(f"  PA: {row.get('pa', '?')}  |  Pitches: {row.get('pitches', '?')}")
    print()
    print(f"  Process+:  {row.get('process_plus', '?'):.1f}")
    print(f"  Decision+: {row.get('decision_plus', '?'):.1f}  (swing/take quality)")
    print(f"  Contact+:  {row.get('contact_plus', '?'):.1f}  (contact execution)")
    print(f"  Power+:    {row.get('power_plus', '?'):.1f}  (xwOBA vs expectation)")
    if 'swing_pct' in row.index:
        print()
        print(f"  Swing%: {row['swing_pct']:.1%}  |  Chase%: {row.get('chase_pct', float('nan')):.1%}")
    if 'xwoba_actual' in row.index:
        print(f"  xwOBA: {row['xwoba_actual']:.3f}  |  vs expected: {row.get('xwoba_vs_expected', 0):+.3f}")

## 10. Decision+ vs Power+ Scatter

Quadrant analysis: elite hitters should appear in top-right (high D+, high P+).
Bottom-right (high P+, low D+): raw power, chases too much.
Top-left (high D+, low P+): disciplined contact, below-average damage.

In [ ]:
if 'decision_plus' in hitters.columns and 'power_plus' in hitters.columns:
    fig, ax = plt.subplots(figsize=(10, 8))

    ax.scatter(hitters['decision_plus'], hitters['power_plus'],
               alpha=0.4, s=20, color='#555', zorder=2)

    # Label top-process hitters
    top_label = hitters.nlargest(15, 'process_plus')
    for _, r in top_label.iterrows():
        name = r.get(name_col, str(r['batter']))
        ax.annotate(name.split()[-1], (r['decision_plus'], r['power_plus']),
                    fontsize=7, alpha=0.85, ha='center', va='bottom')

    # Quadrant lines at 100
    ax.axvline(100, color='gray', linestyle='--', linewidth=0.8)
    ax.axhline(100, color='gray', linestyle='--', linewidth=0.8)

    ax.set_xlabel('Decision+ (swing/take quality)')
    ax.set_ylabel('Power+ (xwOBA above expectation)')
    ax.set_title(f'{YEAR} Decision+ vs Power+ — {len(hitters)} qualified hitters')

    # Quadrant labels
    for (x, y, txt) in [(88, 130, 'Raw power\n(chaser)'), (112, 130, 'Elite'),
                         (88, 72, 'Weak all-around'), (112, 72, 'Disciplined\n(no pop)')]:
        ax.text(x, y, txt, fontsize=8, color='#888', ha='center', style='italic')

    plt.tight_layout()
    plt.show()